### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [1]:
%pip install numpy scikit-learn

### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [3]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [4]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [5]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [6]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [7]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [8]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [9]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [10]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [29]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [11]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [12]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [13]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [14]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [15]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ])

Después vemos a qué documentos corresponden

In [16]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  4703, 10870,  4333])

Obtenemos los 5 documentos más similares:

In [17]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [18]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [19]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [20]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

MultinomialNB()

Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [21]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [22]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


**1. Vectorizar documentos**

In [24]:
#Fijamos la semilla:
np.random.seed(99)

#Armamos un sample:
sample_idx = np.random.choice(len(newsgroups_train.data), 5, replace=False)
print('Índices seleccionados:', sample_idx)

for idx in sample_idx:
    cossim  = cosine_similarity(X_train[idx], X_train)[0]

    # argsort devuelve índices de menor a mayor. invertimos y saltamos el documento mismo (posición 0)
    mostsim = np.argsort(cossim)[::-1][1:6]

    orig_label = newsgroups_train.target_names[y_train[idx]]
    print('\n' + '='*70)
    # Clase original
    print(f'Documento #{idx} — clase: {orig_label}')
    print('-'*70)
    # Extraemos la primera parte del texto para tner de referencia:
    inicio = newsgroups_train.data[idx][:200].replace('\n', '')
    print(f'Texto (inicio): {inicio}')
    print('\n' +'Top-5 más similares:')
    for rank, i in enumerate(mostsim, 1):
        sim_label = newsgroups_train.target_names[y_train[i]]
        acierto = 'Misma clase' if sim_label == orig_label else 'Clase distinta'
        print(f'{rank} - Documento # {i} | sim={cossim[i]:.4f} | {sim_label} [{acierto}]')

Índices seleccionados: [ 3497  7917  3130 10623  6436]

Documento #3497 — clase: comp.sys.mac.hardware
----------------------------------------------------------------------
Texto (inicio): The PowerBook 170 hardware doesn't have a wakeup timer.  Nor does the 140.The Mac Portable had one, and I think the PowerBook 100 had one.  I don'tknow about the newer PowerBooks, but I kind of dou

Top-5 más similares:
1 - Documento # 9428 | sim=0.2596 | comp.sys.mac.hardware [Misma clase]
2 - Documento # 6693 | sim=0.2420 | comp.sys.mac.hardware [Misma clase]
3 - Documento # 4478 | sim=0.2385 | comp.sys.mac.hardware [Misma clase]
4 - Documento # 2 | sim=0.2269 | comp.sys.mac.hardware [Misma clase]
5 - Documento # 5273 | sim=0.2130 | comp.sys.mac.hardware [Misma clase]

Documento #7917 — clase: talk.politics.guns
----------------------------------------------------------------------
Texto (inicio): ]>>]>>> >napalm, then let the wood stove inside ignite it.]>>>                       ^^^^^^^^^^^^^^^^

Al analizar los 5 documentos seleccionados, se observa que la mayoría de los vecinos más similares (entre 3 y 5 de cada 5) pertenecen a la misma clase que el documento original. Esto confirma que la similitud coseno sobre vectores captura bien la cercanía temática, especialmente en clases con vocabulario técnico o específico (por ejemplo, `comp.sys.mac.hardware` o `talk.politics.guns`, donde los 5 vecinos coincidieron en clase en algunos casos).

En los casos donde aparecen vecinos de clase distinta, la mayoría tiene sentido a nivel de contenido: suelen ser clases temáticamente relacionadas que comparten vocabulario de fondo (por ejemplo, `talk.politics.guns` con `talk.politics.mideast`, o `soc.religion.christian` con `alt.atheism`). Esto muestra una limitación esperable del enfoque: al basarse puramente en palabras compartidas, la similitud puede ser alta entre documentos de clases oficialmente distintas si discuten temas cercanos o usan terminología en común.

El método no distingue matices de significado, postura o contexto más allá del vocabulario compartido.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**

In [25]:
#Calculamos la matriz de similaridad para test:
sim_matrix = cosine_similarity(X_test, X_train)
print(f'shape de matriz de similaridad (Test): {sim_matrix.shape}')
#Tiene shape de len(x_test), len(x_train)-> Calcula la similaridad entre cada documento de x_test con x_train

# Para cada documento de test, tomamos el índice de train con mayor similaridad
best_train_idx = np.argmax(sim_matrix, axis=1)

# La predicción es el label del documento de train más similar
y_pred_test = y_train[best_train_idx]

f1= f1_score(y_test, y_pred_test, average='macro')
print(f'F1-Score Macro: {f1:.4f}')

shape de matriz de similaridad (Test): (7532, 11314)
F1-Score Macro: 0.5050


El resultado obtenido de 0.5050 es muy superior al azar (1/20 clases = 0.05), lo que confirma que se capturan señales reales. Sin embargo, este enfoque es un método en que no hay una fase de "entrenamiento" que comprima la información en parámetros. El "modelo" es la matriz X_train completa.
Esto tiene consecuencias costosas, ya que hay que mantener en memoria toda la matriz de entrenamiento (11.314 documentos × 101k términos) y cada vez que se quiere clasificar, hay que volver a calcular la similitud coseno. Este costo crece con el tamaño del set de entrenamiento. En términos prácticos si se agregan 10.000 documentos nuevos al set de entrenamiento, este clasificador se vuelve más lento y pesado. Debido a esto, este enfoque es poco escalable a producción.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

In [26]:
#Vamos a probar 5 configuraciones de TFidVectorizer x 2 Modelos (MultinomialNB y ComplementNB) x 3 hiperparametros c/u (30 configuraciones distintas)
# Grilla de configuraciones del TfId:
TfidfVectorizer_configs = {
    'base'              : TfidfVectorizer(),
    'sublinear_tf'      : TfidfVectorizer(sublinear_tf=True),
    'min_max_df'        : TfidfVectorizer(min_df=2, max_df=0.9),
    'stopwords'         : TfidfVectorizer(stop_words='english'),
    'combinado'         : TfidfVectorizer(sublinear_tf=True, min_df=2, max_df=0.9, stop_words='english'),
}

resultados = []

for vect_name, vect in TfidfVectorizer_configs.items():
    #Fiteamos los vectorizadores:
    Xtr = vect.fit_transform(newsgroups_train.data)
    Xte = vect.transform(newsgroups_test.data)

    #Iteramos para cada modelo:
    for Model, model_name in [(MultinomialNB, 'MultinomialNB'), (ComplementNB, 'ComplementNB')]:
        #Sobre 3 alpha (smoothing parameter) distintos:
        for alpha in [0.01, 0.1, 1.0]:
            clf = Model(alpha=alpha)
            clf.fit(Xtr, y_train)
            f1 = f1_score(y_test, clf.predict(Xte), average='macro')
            resultados.append({'vectorizer': vect_name, 'modelo': model_name, 'alpha': alpha, 'f1_macro': f1})

import pandas as pd
df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados.sort_values('f1_macro', ascending=False)
df_resultados.head(10)

,vectorizer,modelo,alpha,f1_macro
4,base,ComplementNB,0.1,0.695365
23,stopwords,ComplementNB,1.0,0.693611
17,min_max_df,ComplementNB,1.0,0.693482
5,base,ComplementNB,1.0,0.692953
10,sublinear_tf,ComplementNB,0.1,0.692682
29,combinado,ComplementNB,1.0,0.692060
11,sublinear_tf,ComplementNB,1.0,0.692043
22,stopwords,ComplementNB,0.1,0.691919
16,min_max_df,ComplementNB,0.1,0.690643
28,combinado,ComplementNB,0.1,0.687836


El cambio de un modelo one-shot (basado en similaridad coseno) a Naïve Bayes con parámetros optimizados eleva el f1-macro de 0.5050 a 0.6953, demostrando que un modelo entrenado con evidencia estadística supera ampliamente a un enfoque basado en un unico vecino más cercano.

**4. Transponer la matriz documento-término.**

In [27]:
# Transponemos:
X_term_doc = X_train.T

print(f'Matriz documento-término : {X_train.shape}  (documentos × terminos)')
print(f'Matriz término-documento : {X_term_doc.shape}  (terminos × documentos)')

Matriz documento-término : (11314, 101631)  (documentos × terminos)
Matriz término-documento : (101631, 11314)  (terminos × documentos)


In [30]:
# Palabras elegidas manualmente:
palabras_elegidas = ['jesus', 'gun', 'windows', 'space', 'israel']

vocabulario = tfidfvect.vocabulary_

for palabra in palabras_elegidas:
    indice_palabra = vocabulario[palabra]
    vector_palabra = X_term_doc[indice_palabra]
    similitudes    = cosine_similarity(vector_palabra, X_term_doc)[0]
    top_indices    = np.argsort(similitudes)[::-1][1:6]

    print(f'\nPalabra: "{palabra}"')
    print('  Top-5 más similares:')
    for posicion, i in enumerate(top_indices, 1):
        print(f'    {posicion}. {idx2word[i]:<20s}  sim={similitudes[i]:.4f}')


Palabra: "jesus"
  Top-5 más similares:
    1. christ                sim=0.3039
    2. god                   sim=0.2688
    3. kingdom               sim=0.2130
    4. mat                   sim=0.1967
    5. bible                 sim=0.1951

Palabra: "gun"
  Top-5 más similares:
    1. guns                  sim=0.3582
    2. crime                 sim=0.2441
    3. handgun               sim=0.2391
    4. homicides             sim=0.2331
    5. firearms              sim=0.2328

Palabra: "windows"
  Top-5 más similares:
    1. dos                   sim=0.3037
    2. ms                    sim=0.2320
    3. microsoft             sim=0.2219
    4. nt                    sim=0.2140
    5. for                   sim=0.1930

Palabra: "space"
  Top-5 más similares:
    1. nasa                  sim=0.3304
    2. seds                  sim=0.2966
    3. shuttle               sim=0.2928
    4. enfant                sim=0.2803
    5. seti                  sim=0.2465

Palabra: "israel"
  Top-5 más simil

En los 5 casos analizados, las palabras más similares resultaron directamente
relacionadas con el campo temático de la palabra original: "jesus" se asocia a
vocabulario religioso (`christ`, `god`, `bible`), "gun" a vocabulario sobre armas (`guns`, `firearms`, `crime`), "windows" a sistemas operativos (`dos`, `microsoft`), "space" a exploración espacial (`nasa`, `shuttle`) e "israel" a geopolítica de Medio Oriente (`arab`, `syrian`, `lebanon`).

Esto muestra que la matriz término-documento identifica palabras similares por aparecer frecuentemente en los mismos documentos, más que por compartir un significado en sí mismas.